[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/06_linear_regression/04_batch_sgd_minibatch_gd.ipynb)

# Bài 04 — Batch, Stochastic và Mini-batch Gradient Descent

## 1. Nhắc lại về Gradient Descent

Gradient Descent là thuật toán lặp tối ưu hoá dùng trong các bài toán ML/DL. GD lấy đạo hàm của cost function theo $\theta$ và di chuyển **ngược dấu** đạo hàm để tiến về điểm cực tiểu.

<img src="images/gd-2.png" style="width:40%;height:40%;">

Câu hỏi của bài này: **mỗi lần cập nhật $\theta$, ta dùng bao nhiêu dữ liệu để tính gradient?**

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/06_linear_regression

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np

from ml_course.visualization import set_style

set_style()
rng = np.random.default_rng(0)

## 2. Batch Gradient Descent

Từ đầu đến giờ ta đều dùng **Batch Gradient Descent**: dùng **toàn bộ** tập train để tính gradient cho mỗi lần cập nhật. Nhược điểm: với dữ liệu lớn (hàng triệu mẫu), mỗi vòng lặp rất tốn thời gian — và phải duyệt hết dữ liệu chỉ để đi **một** bước.

Ví dụ Linear Regression với $m$ mẫu train:

<img src="images/gd/1.png" style="width:60%;height:60%;">

Batch GD dùng toàn bộ $m$ mẫu cho mỗi bước:

<img src="images/gd/2.png" style="width:60%;height:60%;">

Nếu $m$ = 3 triệu, mỗi lần cập nhật $\theta$ phải tính trên 3 triệu mẫu.

## 3. Stochastic Gradient Descent (SGD)

Mỗi lần cập nhật chỉ dùng **một** điểm dữ liệu $x^{(i)}$ (chọn ngẫu nhiên). Vì loss là **trung bình** trên các điểm, gradient tại một điểm là một **ước lượng không thiên lệch** của gradient đầy đủ: $\mathbb E_i[\nabla \ell_i(\theta)] = \nabla J(\theta)$ — đúng trung bình, nhưng **nhiễu**.

<img src="images/gd/3.png" style="width:60%;height:60%;">

## 4. Mini-batch Gradient Descent

Lấy $k$ mẫu mỗi lần ($1 < k < m$, thường 32–512). Đầu mỗi **epoch** (một lượt duyệt hết dữ liệu), **xáo trộn** dữ liệu rồi chia thành các mini-batch; batch cuối có thể ít hơn $k$ mẫu nếu $m$ không chia hết cho $k$.

```text
batch_size = 1          → Stochastic GD
1 < batch_size < m      → Mini-batch GD
batch_size = m          → Batch GD
```

Ba biến thể chỉ khác nhau ở **một tham số**, nên ta cài đặt bằng **một hàm** *(bổ sung — bản gốc chỉ có mã giả)*:

In [ ]:
def make_data(m=8000, seed=0):
    """Dữ liệu giống bài tập AIVN: (x, y) từ phân phối chuẩn 2 chiều có tương quan cao."""
    r = np.random.default_rng(seed)
    data = r.multivariate_normal([5.0, 6.0], [[1.0, 0.95], [0.95, 1.2]], m)
    X = np.c_[np.ones(m), data[:, 0]]          # thêm cột 1 (bias)
    return X, data[:, 1]


X, y = make_data()
split = int(0.9 * len(y))
X_train, y_train, X_test, y_test = X[:split], y[:split], X[split:], y[split:]


def mse(X, y, theta):
    return np.mean((X @ theta - y) ** 2)


def minibatch_gd(X, y, lr, batch_size, n_epochs=20, seed=0, lr_decay=0.0):
    r = np.random.default_rng(seed)
    m = len(y)
    theta = np.zeros(X.shape[1])
    history = {"epoch_loss": [mse(X, y, theta)], "step_loss": [], "updates": 0}
    for epoch in range(n_epochs):
        idx = r.permutation(m)                              # xáo trộn mỗi epoch
        lr_t = lr / (1 + lr_decay * epoch)                  # (tuỳ chọn) giảm dần learning rate
        for start in range(0, m, batch_size):
            b = idx[start : start + batch_size]
            xb, yb = X[b], y[b]
            grad = xb.T @ (xb @ theta - yb) / len(b)        # gradient TRUNG BÌNH trên batch
            theta -= lr_t * grad
            history["updates"] += 1
            if history["updates"] % max(1, (m // batch_size) // 20) == 0:
                history["step_loss"].append(mse(X, y, theta))
        history["epoch_loss"].append(mse(X, y, theta))
    return theta, history


theta_exact = np.linalg.lstsq(X_train, y_train, rcond=None)[0]
print("Nghiệm chính xác (lstsq):", theta_exact.round(4), "| MSE train tối ưu:", round(mse(X_train, y_train, theta_exact), 4))

> 📝 *Cải tiến so với bản gốc:* gradient được **chia cho kích thước batch** (trung bình) để learning rate có cùng ý nghĩa với mọi batch size; và loss được đo trên **toàn bộ** tập train (thay vì trên từng batch nhỏ) để so sánh công bằng giữa 3 biến thể.

## 5. So sánh tốc độ hội tụ

In [ ]:
settings = {"Batch GD (batch = m)": (len(y_train), 0.05), "Mini-batch (32)": (32, 0.02), "SGD (batch = 1)": (1, 0.005)}
results = {}
for name, (bs, lr) in settings.items():
    start = time.perf_counter()
    theta, hist = minibatch_gd(X_train, y_train, lr=lr, batch_size=bs, n_epochs=20)
    results[name] = (theta, hist, time.perf_counter() - start)
    print(f"{name:22s} θ = {theta.round(4)} | MSE train = {mse(X_train, y_train, theta):.4f} | "
          f"{hist['updates']:6d} lần cập nhật | {results[name][2]:.2f}s")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for name, (theta, hist, secs) in results.items():
    axes[0].plot(hist["epoch_loss"], "o-", ms=3, label=name)
axes[0].axhline(mse(X_train, y_train, theta_exact), color="k", ls=":", label="tối ưu")
axes[0].set_xlabel("epoch (số lượt duyệt hết dữ liệu)")
axes[0].set_ylabel("MSE trên toàn tập train")
axes[0].set_yscale("log")
axes[0].legend()
axes[0].set_title("Theo epoch: SGD/mini-batch giảm loss nhanh hơn nhiều")
for name, (theta, hist, secs) in results.items():
    axes[1].plot(hist["step_loss"][:200], label=name)
axes[1].set_yscale("log")
axes[1].set_xlabel("điểm ghi nhận trong quá trình train")
axes[1].set_title("SGD: đường loss 'răng cưa' do gradient nhiễu")
axes[1].legend()
plt.show()

<img src="images/gd/gd_sgd.png" style="width:55%;height:55%;">

**Nhận xét:**
- Sau 20 epoch, Batch GD mới cập nhật **20 lần** và còn cách xa nghiệm tối ưu (feature $x$ chưa chuẩn hoá nên bài toán khá "méo"); mini-batch và SGD đã rất gần tối ưu.
- Tính theo **epoch**, SGD và mini-batch giảm loss nhanh hơn batch GD rất nhiều — vì trong một epoch chúng cập nhật $\theta$ hàng nghìn lần thay vì một lần.
- SGD **nhiễu**: loss dao động, không bao giờ "đứng yên" hẳn ở cực tiểu với learning rate cố định → cần **giảm dần learning rate** (learning rate decay).
- SGD thuần (batch = 1) tính chậm trên **thời gian thực** vì vòng lặp Python và không tận dụng được vectorization/GPU.
- **Mini-batch** là lựa chọn cân bằng — và là chuẩn trong deep learning: tận dụng vectorization (và GPU), gradient đủ ít nhiễu, cập nhật đủ thường xuyên.

### Learning rate decay cho SGD *(bổ sung)*

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for decay in [0.0, 0.5]:
    _, hist = minibatch_gd(X_train, y_train, lr=0.01, batch_size=1, n_epochs=20, lr_decay=decay)
    ax.plot(hist["step_loss"], label=f"SGD, lr_decay = {decay}")
ax.axhline(mse(X_train, y_train, theta_exact), color="k", ls=":", label="tối ưu")
ax.set_yscale("log")
ax.legend()
ax.set_title("Giảm dần learning rate giúp SGD 'lắng' về gần cực tiểu")
plt.show()

## 6. Chọn batch size *(bổ sung)*

| Batch size | Ưu | Nhược |
|---|---|---|
| Nhỏ (1–16) | cập nhật nhiều, nhiễu có thể giúp thoát điểm yên ngựa / cực tiểu "nhọn" | chậm trên phần cứng, cần LR nhỏ |
| Vừa (32–512) | cân bằng, tận dụng GPU | — |
| Lớn (hàng nghìn+) | gradient chính xác, tận dụng song song tối đa | ít cập nhật/epoch, cần tăng LR, đôi khi generalize kém hơn |

Quy tắc kinh nghiệm: bắt đầu với 32–256; khi tăng batch size $k$ lần, thử tăng learning rate cũng khoảng $k$ lần (linear scaling rule).

## 7. Học với dữ liệu lớn *(bổ sung — chủ đề "Learning with Large Datasets" của khoá AIVN, tuần 5)*

Khi có hàng chục triệu mẫu, batch GD không còn khả thi. Bốn câu hỏi thực tế: có thật cần nhiều dữ liệu không? Theo dõi hội tụ thế nào khi tính loss trên toàn bộ dữ liệu cũng đã quá đắt? Làm gì khi dữ liệu **đến liên tục**? Và chia việc cho nhiều máy ra sao?

### 7.1 Trước hết: có thật cần nhiều dữ liệu?

Trước khi tốn công thu thập/xử lý 100 triệu mẫu, hãy vẽ **learning curve** với một tập nhỏ ([chương 05 · bài 05](../05_machine_learning_fundamentals/05_learning_curves_and_error_analysis.ipynb)). Nếu sai số train và validation đã sát nhau và cùng cao (**high bias**), thêm dữ liệu gần như vô ích — cần mô hình mạnh hơn hoặc feature tốt hơn trước. Chỉ khi có khoảng cách lớn giữa hai đường (**high variance**) thì thêm dữ liệu mới đáng giá.

### 7.2 Theo dõi hội tụ của SGD mà không tính loss trên toàn bộ dữ liệu

Ở trên ta đo MSE trên **toàn bộ** tập train sau mỗi đoạn — với 100 triệu mẫu thì việc đo đó đắt hơn cả việc train. Mẹo của Andrew Ng: với mỗi mẫu, tính loss **trước khi** dùng nó để cập nhật (gần như miễn phí vì phải tính sai số để lấy gradient), rồi vẽ **trung bình của 500 mẫu gần nhất**. Vì mẫu chưa được dùng để cập nhật, đây là ước lượng trung thực của loss hiện tại.

In [ ]:
def sgd_with_monitor(X, y, lr, window=500, n_epochs=2, seed=0):
    r = np.random.default_rng(seed)
    theta = np.zeros(X.shape[1])
    recent, cheap_curve, full_curve = [], [], []
    for _ in range(n_epochs):
        for i in r.permutation(len(y)):
            err = X[i] @ theta - y[i]
            recent.append(err**2)                    # loss của mẫu i TRƯỚC khi cập nhật
            theta -= lr * err * X[i]
            if len(recent) == window:
                cheap_curve.append(np.mean(recent))  # chi phí: O(1) mỗi mẫu
                full_curve.append(mse(X, y, theta))  # chỉ để đối chiếu — chi phí O(m) mỗi lần
                recent = []
    return theta, cheap_curve, full_curve


_, cheap_curve, full_curve = sgd_with_monitor(X_train, y_train, lr=0.005)
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(cheap_curve, "o-", ms=3, label="trung bình loss 500 mẫu gần nhất (rẻ)")
ax.plot(full_curve, "s-", ms=3, label="MSE trên toàn bộ tập train (đắt)")
ax.set_yscale("log")
ax.set_xlabel("số khối 500 lần cập nhật")
ax.legend()
ax.set_title("Đường 'rẻ' bám sát đường 'đắt' — đủ để biết SGD có hội tụ không")
plt.show()
print(f"khối cuối: ước lượng rẻ = {cheap_curve[-1]:.4f} | MSE đầy đủ = {full_curve[-1]:.4f}")

Đường "rẻ" nhiễu hơn nhưng cho cùng thông tin: loss có giảm không, đã phẳng chưa. Nếu nó **tăng** dần → learning rate quá lớn; nếu quá nhiễu để đọc → tăng `window`.

### 7.3 Online learning: học từ luồng dữ liệu

Nhiều hệ thống nhận dữ liệu **liên tục** (giao dịch, lượt click, giá thị trường) và quan hệ $x \to y$ **trôi dần** theo thời gian. Thay vì train lại từ đầu, **online learning** cập nhật mô hình bằng từng lô dữ liệu mới rồi bỏ lô đó đi — không cần lưu toàn bộ lịch sử. Trong scikit-learn, các mô hình học bằng SGD có `partial_fit`.

Mô phỏng: mỗi ngày có 200 giao dịch; giá nền tăng 0.05 đơn vị/ngày (thị trường trượt). Đánh giá kiểu **"test-then-train"**: dự đoán lô của ngày mới **trước**, ghi sai số, **rồi** mới học từ lô đó.

In [ ]:
from sklearn.linear_model import SGDRegressor


def day_batch(day, n=200):
    r = np.random.default_rng(1000 + day)
    x = r.uniform(0, 10, n)
    y_day = (2 + 0.05 * day) + 3 * x + r.normal(0, 1, n)   # giá nền trượt theo ngày
    return x[:, None], y_day


static = SGDRegressor(learning_rate="constant", eta0=0.005, random_state=0)
online = SGDRegressor(learning_rate="constant", eta0=0.005, random_state=0)
for day in range(10):                       # 10 ngày đầu: cả hai cùng học
    Xd, yd = day_batch(day)
    for _ in range(5):
        static.partial_fit(Xd, yd)
        online.partial_fit(Xd, yd)

days, err_static, err_online = range(10, 120), [], []
for day in days:
    Xd, yd = day_batch(day)
    err_static.append(np.mean(np.abs(static.predict(Xd) - yd)))   # test trước…
    err_online.append(np.mean(np.abs(online.predict(Xd) - yd)))
    online.partial_fit(Xd, yd)                                      # …rồi mới train (chỉ mô hình online)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(days, err_static, label="huấn luyện một lần rồi đứng yên")
ax.plot(days, err_online, label="online: partial_fit mỗi ngày")
ax.set_xlabel("ngày")
ax.set_ylabel("MAE trên lô của ngày đó")
ax.legend()
ax.set_title("Dữ liệu trôi: mô hình đứng yên tệ dần, mô hình online bám theo")
plt.show()
print(f"MAE 10 ngày cuối: đứng yên = {np.mean(err_static[-10:]):.2f} | online = {np.mean(err_online[-10:]):.2f} "
      f"(nhiễu thuần có MAE ≈ {np.sqrt(2 / np.pi):.2f})")

Mô hình online gần như giữ sai số ở mức nhiễu, còn mô hình đứng yên tệ dần theo độ trôi. Đánh đổi: online learning phản ứng nhanh với thay đổi nhưng cũng **dễ bị kéo lệch** bởi dữ liệu bẩn hoặc tấn công — trong thực tế luôn kèm giám sát (chương 40) và khả năng quay về phiên bản cũ.

### 7.4 Map-reduce và song song dữ liệu

Gradient của batch GD là một **tổng trên các mẫu**:

$$\nabla J(\theta) = \frac{1}{m}\sum_{i=1}^{m}\big(h_\theta(x^{(i)}) - y^{(i)}\big)\,x^{(i)} = \frac{1}{m}\Big(\underbrace{\textstyle\sum_{i \in \text{phần } 1}}_{\text{máy 1}} + \dots + \underbrace{\textstyle\sum_{i \in \text{phần } 4}}_{\text{máy 4}}\Big)$$

Nên có thể chia dữ liệu cho 4 máy: mỗi máy tính tổng trên phần của mình (**map**), một máy cộng 4 kết quả (**reduce**). Kết quả giống **hệt** tính trên một máy:

In [ ]:
from concurrent.futures import ThreadPoolExecutor


def partial_gradient_sum(shard, theta):
    """'map': mỗi máy chỉ nhìn thấy phần dữ liệu của mình."""
    X_part, y_part = shard
    return X_part.T @ (X_part @ theta - y_part)


theta_now = np.array([0.5, 0.5])
shards = list(zip(np.array_split(X_train, 4), np.array_split(y_train, 4)))     # 4 "máy"
with ThreadPoolExecutor(max_workers=4) as pool:
    partial_sums = list(pool.map(partial_gradient_sum, shards, [theta_now] * 4))
grad_mapreduce = np.sum(partial_sums, axis=0) / len(y_train)                    # 'reduce'
grad_single = X_train.T @ (X_train @ theta_now - y_train) / len(y_train)
print("map-reduce :", grad_mapreduce.round(6))
print("một máy    :", grad_single.round(6), "| giống nhau:", np.allclose(grad_mapreduce, grad_single))

Ý tưởng này có mặt ở khắp nơi:
- **Spark MLlib, Dask**: dữ liệu nằm rải trên nhiều máy, các phép tổng (gradient, thống kê, đếm của Naive Bayes — chương 09) được tính bằng map-reduce.
- **Huấn luyện deep learning song song dữ liệu** (PyTorch `DistributedDataParallel`): mỗi GPU tính gradient trên một phần của mini-batch, rồi **all-reduce** lấy trung bình — đúng phép "reduce" ở trên.
- Thuật toán **không** là tổng độc lập trên các mẫu (vd SGD tuần tự từng mẫu) thì khó song song hoá theo cách này.

## 🧠 Kiểm tra nhanh

1. Có 1 triệu mẫu, batch size 1 000. Một epoch có bao nhiêu lần cập nhật $\theta$?
2. Vì sao gradient trên một mini-batch ngẫu nhiên là ước lượng "không thiên lệch" của gradient đầy đủ?
3. Vì sao phải xáo trộn dữ liệu ở **mỗi** epoch?
4. Loss của SGD dao động mãi không giảm thêm — nên làm gì?
5. Với 100 triệu mẫu, làm sao biết SGD có đang hội tụ mà không phải tính loss trên toàn bộ dữ liệu?
6. Online learning khác việc train lại từ đầu mỗi ngày ở điểm nào? Rủi ro của nó là gì?
7. Vì sao gradient của batch GD tính được bằng map-reduce, còn SGD từng mẫu thì khó song song hoá theo cách đó?

## Tài liệu tham khảo

[1] CS230 — Deep Learning (Stanford) · [2] [Biến thể của Gradient Descent — Machine Learning Cơ Bản](https://machinelearningcoban.com/2017/01/16/gradientdescent2/) · [3] [Difference between Batch Gradient Descent and Stochastic Gradient Descent](https://towardsdatascience.com/difference-between-batch-gradient-descent-and-stochastic-gradient-descent-1187f1291aa1) · [4] Andrew Ng — *Machine Learning* (Coursera), tuần 10: Large Scale Machine Learning · [5] [scikit-learn — Strategies to scale computationally: bigger data](https://scikit-learn.org/stable/computing/scaling_strategies.html)

---
<sub>Bài gốc: *Stochastic, Mini-Batch, Batch Gradient Descent* — [AIVN-Machine-Learning](https://github.com/thanhhff/AIVN-Machine-Learning), Tuần 5 (MIT License). Đã bổ sung: cài đặt đầy đủ (bản gốc là mã giả), so sánh công bằng theo epoch và thời gian, learning rate decay, chọn batch size, học với dữ liệu lớn (theo dõi hội tụ SGD, online learning, map-reduce — phần "Learning with Large Datasets" mà bản gốc chưa có tài liệu).</sub>